## 准备数据

In [4]:
import os
os.environ['TF_XLA_FLAGS'] = '--tf_xla_enable_xla_devices=false'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers, datasets

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # or any {'0', '1', '2'}

def mnist_dataset():
    (x, y), (x_test, y_test) = datasets.mnist.load_data()
    #normalize
    x = x/255.0
    x_test = x_test/255.0
    
    return (x, y), (x_test, y_test)

In [5]:
print(list(zip([1, 2, 3, 4], ['a', 'b', 'c', 'd'])))

[(1, 'a'), (2, 'b'), (3, 'c'), (4, 'd')]


## 建立模型

In [6]:
import tensorflow as tf
from tensorflow.keras import optimizers

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("成功开启GPU显存按需分配")
    except RuntimeError as e:
        print(e)

class myModel:
    def __init__(self):
        self.W1 = tf.Variable(tf.random.truncated_normal([28 * 28, 256], stddev=0.1))
        self.b1 = tf.Variable(tf.zeros([256]))
        
        self.W2 = tf.Variable(tf.random.truncated_normal([256, 10], stddev=0.1))
        self.b2 = tf.Variable(tf.zeros([10]))
        
    def __call__(self, x):
        x_flatten = tf.reshape(x, [-1, 28 * 28])
        h1 = tf.matmul(x_flatten, self.W1) + self.b1
        h1_relu = tf.nn.relu(h1)
        logits = tf.matmul(h1_relu, self.W2) + self.b2
        return logits
        
model = myModel()
optimizer = optimizers.Adam()
print("模型和优化器初始化成功")

成功开启GPU显存按需分配
模型和优化器初始化成功


## 计算 loss

In [7]:
@tf.function
def compute_loss(logits, labels):
    return tf.reduce_mean(
        tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels))

@tf.function
def compute_accuracy(logits, labels):
    predictions = tf.argmax(logits, axis=1)
    return tf.reduce_mean(tf.cast(tf.equal(predictions, labels), tf.float32))

@tf.function
def train_one_step(model, optimizer, x, y):
    with tf.GradientTape() as tape:
        logits = model(x)
        loss = compute_loss(logits, y)

    trainable_vars = [model.W1, model.W2, model.b1, model.b2]
    grads = tape.gradient(loss, trainable_vars)
    
    optimizer.apply_gradients(zip(grads, trainable_vars))

    accuracy = compute_accuracy(logits, y)
    return loss, accuracy

@tf.function
def test(model, x, y):
    logits = model(x)
    loss = compute_loss(logits, y)
    accuracy = compute_accuracy(logits, y)
    return loss, accuracy

## 实际训练

In [8]:
train_data, test_data = mnist_dataset()
for epoch in range(50):
    loss, accuracy = train_one_step(model, optimizer, 
                                    tf.constant(train_data[0], dtype=tf.float32), 
                                    tf.constant(train_data[1], dtype=tf.int64))
    print('epoch', epoch, ': loss', loss.numpy(), '; accuracy', accuracy.numpy())
loss, accuracy = test(model, 
                      tf.constant(test_data[0], dtype=tf.float32), 
                      tf.constant(test_data[1], dtype=tf.int64))

print('test loss', loss.numpy(), '; accuracy', accuracy.numpy())

epoch 0 : loss 2.5528245 ; accuracy 0.074533336
epoch 1 : loss 2.2871366 ; accuracy 0.14156666
epoch 2 : loss 2.0772727 ; accuracy 0.24826667
epoch 3 : loss 1.8916948 ; accuracy 0.37723333
epoch 4 : loss 1.7201968 ; accuracy 0.50528336
epoch 5 : loss 1.5613581 ; accuracy 0.6047
epoch 6 : loss 1.4162246 ; accuracy 0.66721666
epoch 7 : loss 1.2856863 ; accuracy 0.7092
epoch 8 : loss 1.1698053 ; accuracy 0.7408
epoch 9 : loss 1.0678807 ; accuracy 0.76523334
epoch 10 : loss 0.97889465 ; accuracy 0.7841
epoch 11 : loss 0.90160835 ; accuracy 0.7991667
epoch 12 : loss 0.8346877 ; accuracy 0.8124167
epoch 13 : loss 0.7766922 ; accuracy 0.8222333
epoch 14 : loss 0.72613543 ; accuracy 0.83038336
epoch 15 : loss 0.68178684 ; accuracy 0.8365833
epoch 16 : loss 0.6427417 ; accuracy 0.84171665
epoch 17 : loss 0.60841167 ; accuracy 0.84705
epoch 18 : loss 0.5783559 ; accuracy 0.8522
epoch 19 : loss 0.5520878 ; accuracy 0.8568
epoch 20 : loss 0.5290251 ; accuracy 0.86085
epoch 21 : loss 0.50859636 ; a